# Web Scraping de Noticias: Análisis Comparativo de Medios Costarricenses

## Tema
Extracción avanzada de noticias de múltiples medios de comunicación costarricenses utilizando APIs de búsqueda,
procesamiento de contenido HTML con trafilatura, y análisis de sentimiento a escala.

## Objetivos de Aprendizaje
1. **Implementar búsqueda estructurada** mediante Google News RSS con operadores `site:` para precisión quirúrgica
2. **Resolver URLs ocultas** en redirecciones de Google Search para acceder a contenido original
3. **Extraer contenido HTML completo** usando trafilatura para análisis más profundo que solo títulos
4. **Traducir contenido automáticamente** para análisis de sentimiento multilingüe con deep_translator
5. **Crear visualizaciones comparativas** (heatmaps, gráficos de dispersión) para análisis editorial

## Requisitos Previos
- Experiencia con web scraping básico
- Conocimiento de pandas para manipulación de datos
- Familiaridad con matplotlib y seaborn para visualización
- Comprensión de análisis de sentimiento y subjetividad
- Conexión a internet confiable

## Duración Estimada
3-4 horas de clase + 3 horas de ejercicios prácticos

## Nota Importante
**Este notebook REQUIERE conexión activa a internet.** El web scraping depende de:
- Acceso a Google News RSS
- Disponibilidad de sitios web objetivo
- Capacidad de resolver redirects de Google
- APIs de traducción (Google Translate a través de deep_translator)

Los sitios pueden estar bloqueados por CAPTCHA o cambios de estructura HTML.
Se implementan pausas aleatorias (1-2 segundos) entre solicitudes para evitar bloqueo.



## 1. Introduccion: Web Scraping Avanzado

Este notebook demuestra tecnicas profesionales de web scraping para:
- Buscar de forma precisa en Google News RSS
- Resolver URLs enmascaradas por Google
- Extraer contenido completo (no solo titulos)
- Analizar multiples medios en paralelo
- Comparar sesgo editorial entre medios

### Diferencia con Web Scraping Basico
- **Basico**: Extrae HTML directo de sitios (puede fallar con cambios)
- **Avanzado**: Usa APIs (RSS, APIs publicas) y maneja redirecciones

### Flujo del Analisis
1. Definir medios objetivo y temas de busqueda
2. Realizar busquedas quirurgicas por sitio con `site:` operator
3. Resolver URLs de Google a URLs reales
4. Extraer contenido completo con trafilatura
5. Traducir contenido y analizar sentimiento
6. Visualizar comparativas entre medios



## 2. Configuracion e Importaciones

Instalamos las librerias requeridas y configuramos parametros globales.


In [ ]:
# Instalacion de dependencias (ejecutar si es necesario)
import subprocess
import sys

# Lista de paquetes necesarios
required_packages = ['feedparser', 'trafilatura', 'deep_translator', 'lxml']

for package in required_packages:
    try:
        __import__(package)
    except ImportError:
        print(f"Instalando {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

print("OK: Todas las dependencias estan instaladas")


In [ ]:
import feedparser
import trafilatura
import requests
from textblob import TextBlob
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import numpy as np
import pandas as pd
from deep_translator import GoogleTranslator
from lxml import html
from urllib.parse import quote
import time
import random
from typing import List, Dict, Optional, Tuple

# Configuracion de matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("OK: Importaciones completadas exitosamente")



## 3. Parametros Globales y Configuracion

Definimos los medios objetivo, keywords y parametros de red.


In [ ]:
# --- CONFIGURACION GLOBAL ---

# Inicializar traductor (Google Translate)
try:
    translator_en = GoogleTranslator(source='auto', target='en')
    print("OK: Traductor inicializado")
except Exception as e:
    print(f"Warning: Error al inicializar traductor: {e}")
    translator_en = None

# Headers HTTP para imitar navegador
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Accept-Language': 'es-419,es;q=0.9'
}

# Cookies para consentimiento GDPR
COOKIES = {'CONSENT': 'YES+CB'}

# Parametros de red
REQUEST_TIMEOUT = 10  # segundos
PAUSE_MIN = 1.0  # pausa minima entre solicitudes
PAUSE_MAX = 2.0  # pausa maxima entre solicitudes

# Estrategia: Seleccionar medios "gigantes" que garantizan cobertura
MEDIOS_OBJETIVO = [
    "nacion.com",          # La Nacion
    "crhoy.com",           # CRHoy
    "teletica.com",        # Teletica
    "monumental.co.cr",    # Monumental
    "delfino.cr",          # Delfino
    "semanariouniversidad.com"  # Semanario Universidad
]

# Temas a investigar (maximo 5 para no saturar)
TEMAS = ['Gobierno', 'Seguridad', 'Economia', 'Corrupcion', 'Salud']

print(f"Configuracion:")
print(f"  Medios: {len(MEDIOS_OBJETIVO)}")
print(f"  Temas: {len(TEMAS)}")
print(f"  Total de busquedas: {len(MEDIOS_OBJETIVO) * len(TEMAS)}")



## 4. Funciones Principales

Implementamos las funciones clave del pipeline de scraping.


In [ ]:
def obtener_url_real(google_url: str) -> str:
    """Resuelve una URL de Google News a la URL real del articulo.

    Google News encubre las URLs reales con redirecciones. Esta funcion
    sigue las redirecciones para obtener la URL original del sitio.

    Args:
        google_url (str): URL que comienza con https://news.google.com

    Returns:
        str: URL real del articulo, o la URL original si no se puede resolver
    """
    try:
        response = requests.get(
            google_url,
            headers=HEADERS,
            cookies=COOKIES,
            timeout=REQUEST_TIMEOUT,
            allow_redirects=True
        )

        # Si la redireccion funciono, retornar URL final
        if "google.com" not in response.url:
            return response.url

        # Fallback: parsear HTML para encontrar enlaces
        tree = html.fromstring(response.content)
        links = tree.xpath('//a/@href')

        for link in links:
            if link.startswith('http') and 'google.com' not in link:
                return link

        return google_url

    except requests.Timeout:
        print(f"  Warning: Timeout resolviendo {google_url[:40]}...")
        return google_url
    except Exception as e:
        print(f"  Warning: Error resolviendo URL: {type(e).__name__}")
        return google_url


def busqueda_quirurgica(keyword: str, medio: str) -> List[Dict]:
    """Busca noticias EXCLUSIVAMENTE en un medio especifico.

    Utiliza el operador `site:` de Google para busqueda de precision.
    Ejemplo: site:nacion.com Corrupcion

    Args:
        keyword (str): Tema o palabra clave a buscar
        medio (str): Dominio del medio (ej: "nacion.com")

    Returns:
        List[Dict]: Lista con un dict por articulo encontrado
    """
    items = []

    # Construir query de busqueda
    query = f'site:{medio} {keyword}'
    encoded = quote(query)

    # URL del RSS de Google News con la busqueda
    rss_url = (
        f"https://news.google.com/rss/search?"
        f"q={encoded}&hl=es-419&gl=CR&ceid=CR:es"
    )

    try:
        # Parsear feed RSS
        feed = feedparser.parse(rss_url)

        # Tomar solo la primera (mas reciente) para evitar duplicados
        if feed.entries:
            entry = feed.entries[0]

            # Resolver URL real
            real_url = obtener_url_real(entry.link)

            # Crear dict con informacion extraida
            items.append({
                "Medio": medio,
                "Tema": keyword,
                "Titulo": entry.title if hasattr(entry, 'title') else "N/A",
                "Link": real_url,
                "Resumen": entry.description if hasattr(entry, 'description') else ""
            })

    except Exception as e:
        print(f"  Error en {medio}: {type(e).__name__}: {str(e)[:40]}")

    return items


def analizar_lote(items: List[Dict]) -> pd.DataFrame:
    """Analiza un lote de articulos extraidos.

    Para cada articulo:
    1. Intenta descargar contenido completo con trafilatura
    2. Si falla, usa el resumen de Google News
    3. Traduce a ingles (TextBlob funciona mejor con ingles)
    4. Analiza sentimiento con TextBlob
    5. Calcula subjetividad

    Args:
        items (List[Dict]): Lista de articulos extraidos

    Returns:
        pd.DataFrame: DataFrame con analisis de sentimiento
    """
    data = []
    total = len(items)
    print(f"\n--- Analizando {total} articulos extraidos ---\n")

    for i, item in enumerate(items, 1):
        # Mostrar progreso
        print(f"[{i}/{total}] {item['Medio'][:12]:<12} | {item['Tema'][:12]:<12} | "
              f"{item['Titulo'][:40]:<40}", end="\r")

        texto = ""

        # 1. Intentar descargar contenido completo
        try:
            downloaded = trafilatura.fetch_url(item['Link'])
            if downloaded:
                extracted = trafilatura.extract(downloaded)
                if extracted and len(extracted) > 200:
                    texto = extracted
        except:
            pass

        # 2. Si falla, usar resumen de Google News
        if not texto:
            try:
                clean_sum = html.fromstring(item['Resumen']).text_content() if item['Resumen'] else ""
                texto = f"{item['Titulo']}. {clean_sum}"
            except:
                texto = item['Titulo']

        # 3. Analizar sentimiento
        if texto and len(texto.strip()) > 10:
            try:
                # Traducir a ingles (limitar a 2000 caracteres por velocidad)
                if translator_en:
                    txt_en = translator_en.translate(texto[:2000])
                else:
                    txt_en = texto[:2000]

                # Analizar con TextBlob
                blob = TextBlob(txt_en)

                data.append({
                    "Medio": item['Medio'],
                    "Tema": item['Tema'],
                    "Score": blob.sentiment.polarity,           # -1 a +1
                    "Subjetividad": blob.sentiment.subjectivity,  # 0 a 1
                    "Titulo": item['Titulo']
                })
            except Exception as e:
                print(f"\n  Warning: Error analizando sentimiento: {type(e).__name__}")

    print(f"\nOK: Analisis completado: {len(data)} articulos procesados")
    return pd.DataFrame(data)


In [ ]:
def visualizar_comparativa(df: pd.DataFrame) -> None:
    """Crea visualizaciones comparativas del analisis de medios.

    Genera 4 graficos:
    1. Heatmap: Sentimiento por Medio x Tema
    2. Barras: Ranking de positividad/negatividad general
    3. Dispersion: Sentimiento vs Subjetividad (sesgo editorial)
    4. Tabla de resultados detallados

    Args:
        df (pd.DataFrame): DataFrame con analisis de sentimiento
    """
    if df.empty:
        print("Error: No hay datos para visualizar.")
        return

    # Crear figura con subplots
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)
    plt.suptitle(
        "Analisis Comparativo de Medios Costarricenses",
        fontsize=20, fontweight='bold'
    )

    # 1. HEATMAP: Sentimiento por Medio y Tema
    ax1 = fig.add_subplot(gs[0, :])

    pivot_table = df.pivot_table(
        index='Medio',
        columns='Tema',
        values='Score',
        aggfunc='mean'
    )

    sns.heatmap(
        pivot_table,
        annot=True,
        fmt='.2f',
        cmap="RdYlGn",
        center=0,
        vmin=-0.6,
        vmax=0.6,
        linewidths=1,
        ax=ax1,
        cbar_kws={'label': 'Sentimiento Promedio'}
    )
    ax1.set_title(
        "Mapa de Calor: Sesgo Medio x Tema (negativo[rojo] a positivo[verde])",
        fontsize=14, fontweight='bold'
    )
    ax1.set_xlabel("Tema", fontweight='bold')
    ax1.set_ylabel("Medio", fontweight='bold')

    # 2. BARRAS: Sentimiento promedio por medio
    ax2 = fig.add_subplot(gs[1, 0])
    medios_avg = df.groupby('Medio')['Score'].mean().sort_values()
    colors = ['#d62728' if x < 0 else '#2ca02c' for x in medios_avg.values]
    medios_avg.plot(kind='barh', color=colors, ax=ax2, edgecolor='black')
    ax2.set_title("Sentimiento Promedio por Medio", fontsize=12, fontweight='bold')
    ax2.set_xlim(-0.5, 0.5)
    ax2.axvline(0, color='black', linewidth=1.5, linestyle='--', alpha=0.7)
    ax2.set_xlabel("Score Promedio", fontweight='bold')

    # 3. DISPERSION: Sentimiento vs Subjetividad
    ax3 = fig.add_subplot(gs[1, 1])
    scatter = sns.scatterplot(
        data=df,
        x='Score',
        y='Subjetividad',
        hue='Medio',
        style='Tema',
        s=150,
        ax=ax3,
        alpha=0.7,
        edgecolor='black',
        linewidth=0.5
    )
    ax3.axvline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax3.axhline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax3.set_title("Analisis de Sesgo Editorial", fontsize=12, fontweight='bold')
    ax3.set_xlabel("Sentimiento (Negativo <- -> Positivo)", fontweight='bold')
    ax3.set_ylabel("Subjetividad (0=Objetivo, 1=Subjetivo)", fontweight='bold')
    ax3.legend(bbox_to_anchor=(1.05, 1), loc='upper left', ncol=1, fontsize=8)
    ax3.set_xlim(-0.6, 0.6)
    ax3.set_ylim(-0.05, 1.05)

    plt.tight_layout()
    plt.savefig('analisis_medios.png', dpi=300, bbox_inches='tight')
    print("\nOK: Graficos guardados en: analisis_medios.png")
    plt.show()

    # TABLA DE RESULTADOS DETALLADOS
    print("\n" + "="*100)
    print(f"{'MEDIO':<18} | {'TEMA':<15} | {'SCORE':>6} | {'SUBJ':>4} | {'TITULO':<40}")
    print("="*100)

    for _, row in df.iterrows():
        score_symbol = "+" if row['Score'] > 0 else "-" if row['Score'] < 0 else "="
        print(
            f"{row['Medio'][:18]:<18} | "
            f"{row['Tema'][:15]:<15} | "
            f"{score_symbol}{abs(row['Score']):>5.2f} | "
            f"{row['Subjetividad']:>4.2f} | "
            f"{row['Titulo'][:40]:<40}"
        )

    print("="*100)



## 5. Ejecucion Principal

Ejecutamos el pipeline completo de busqueda y analisis.


In [ ]:
def main():
    """Funcion principal que ejecuta todo el pipeline."""

    print("="*80)
    print("ANALISIS COMPARATIVO DE MEDIOS COSTARRICENSES")
    print("="*80)
    print(f"Medios: {len(MEDIOS_OBJETIVO)}")
    print(f"Temas: {len(TEMAS)}")
    print(f"Total de busquedas: {len(MEDIOS_OBJETIVO) * len(TEMAS)}")
    print("="*80)

    all_items = []

    # Bucle doble: Tema x Medio (garantiza cobertura)
    total_queries = len(TEMAS) * len(MEDIOS_OBJETIVO)
    count = 0

    print(f"\n>> Ejecutando {total_queries} busquedas de precision...\n")

    for tema in TEMAS:
        print(f"Buscando sobre '{tema}':")

        for medio in MEDIOS_OBJETIVO:
            count += 1
            res = busqueda_quirurgica(tema, medio)

            if res:
                print(f"   [OK] {medio}: Encontrado")
                all_items.extend(res)
            else:
                print(f"   [  ] {medio}: Sin noticias recientes")

            # Pausa aleatoria para evitar bloqueo por Google
            pausa = random.uniform(PAUSE_MIN, PAUSE_MAX)
            time.sleep(pausa)

    print(f"\n{'='*80}")
    print(f"Total de articulos encontrados: {len(all_items)}")
    print(f"{'='*80}")

    # Analizar si se encontraron articulos
    if all_items:
        df = analizar_lote(all_items)

        if not df.empty:
            # Guardar resultados
            df.to_csv('resultados_analisis_medios.csv', index=False, encoding='utf-8')
            print(f"\nOK: Resultados guardados en: resultados_analisis_medios.csv")

            # Visualizar
            visualizar_comparativa(df)
            return df
        else:
            print("\nWarning: No se pudieron analizar los articulos encontrados.")
            return None
    else:
        print("\nError: No se encontraron articulos.")
        print("Posibles causas:")
        print("   - Google esta pidiendo CAPTCHA (cambiar IP/proxy)")
        print("   - Problemas de conectividad")
        print("   - Medios no tienen noticias sobre estos temas")
        return None


if __name__ == "__main__":
    df_resultados = main()



## 6. Resumen de Analisis

### Metodologia
1. **Busqueda Quirurgica**: Usando operador `site:` en Google RSS
2. **Resolucion de URLs**: Descifrando redirecciones de Google
3. **Extraccion de Contenido**: Con trafilatura para texto completo
4. **Analisis de Sentimiento**: Traduccion a ingles + TextBlob
5. **Visualizacion**: Heatmaps y scatter plots para comparativas

### Interpretacion de Resultados
- **Heatmap**: Verde = positivo, Rojo = negativo. Muestra sesgo por tema
- **Barras**: Ranking general de positividad/negatividad
- **Dispersion**: Cuadrantes indican objetividad vs sesgo editorial
  - Arriba-izquierda: Opiniones negativas
  - Arriba-derecha: Opiniones positivas
  - Abajo-izquierda: Noticias negativas
  - Abajo-derecha: Noticias positivas

### Limitaciones
- TextBlob es basico; existen modelos BERT mas avanzados
- Traduccion automatica puede perder matices
- Google puede bloquear busquedas masivas (implementar proxy si es necesario)
- Algunos articulos no se descargan correctamente
- Analisis de sentimiento es mejor en ingles

---

## 7. Ejercicios Practicos

### Ejercicio 1: Agregar Nuevos Medios
**Objetivo**: Expandir analisis a otros medios

Modifica `MEDIOS_OBJETIVO` para incluir:
- radiocastella.co.cr
- albareport.com
- eleconomista.co.cr

Ejecuta el analisis nuevamente y compara resultados.

### Ejercicio 2: Analisis Temporal Avanzado
**Objetivo**: Visualizar como cambia sentimiento en el tiempo

Modifica `busqueda_quirurgica()` para retornar multiples resultados (no solo el primero).
Luego extrae la fecha de publicacion y crea un grafico de serie temporal:
- Eje X: Fecha
- Eje Y: Score promedio de sentimiento
- Lineas: Una por cada tema

### Ejercicio 3: Deteccion de Polarizacion
**Objetivo**: Identificar que medios son mas "sensacionalistas"

Calcula para cada medio:
1. Varianza del sentimiento (dispersion de opiniones)
2. Varianza de la subjetividad (uso de opinion vs hechos)
3. Correlacion entre tema y sentimiento

Crea un grafico de "Indice de Polarizacion" por medio.

---

## Recursos y Referencias
- [Google News RSS API](https://news.google.com/)
- [Trafilatura: Content Extraction](https://trafilatura.readthedocs.io/)
- [Feedparser: RSS Parsing](https://feedparser.readthedocs.io/)
- [TextBlob: Sentiment Analysis](https://textblob.readthedocs.io/)
- [Deep Translator: Translation](https://github.com/nidhaloff/deep-translator)
